# Google Colab
This notebook is intended to be executed on Google Golab.

In [29]:
# setup for Google Colab
!pip install torch torchvision
!pip install requests
!pip install tqdm

In [30]:
# imports
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import requests
import subprocess
import json

# MyDataset class
MyDataset class is a child of the pytorch Dataset class that provides access to tinyimagenet and places2 datasets.  
MyDataset class provide the posibility to switch from one dataset to another and from one split to another(train -> val, val -> train).  

Tinyimagenet was downloaded from [http://cs231n.stanford.edu/tiny-imagenet-200.zip](http://cs231n.stanford.edu/tiny-imagenet-200.zip), and places2 was downloaded from [https://www.kaggle.com/datasets/nickj26/places2-mit-dataset](https://www.kaggle.com/datasets/nickj26/places2-mit-dataset).  

For the tinyimagenet dataset i did the following:
- Downloaded the original archive from the provided link.
- Unpacked the archive locally.
- Originally, the dataset consists of two big train, val folders with many subfolders. For each train and val folder, i did not want to have subfolders, so i moved the images from the subfolders into the root train/val big folders and deleted all the subfolders. The final result are just the two big train and val folders with images inside, no subfolders.
- For each train, val, big folders i renamed the images with the following patters: 0.extension, 1.extension, 2.extension, etc. The reason behind is to make indexing easier.
Ex: for train: 0.extension, 1.extension, 2.extension,etc; for val: 0.extension, 1.extension, 2.extension. etc.
- Added train_index.json, val_index.json with a 'files' dictionary were key = the index('0', '1', '2','3', etc.), value = relative path to image('0':'train/0.jpeg for train_index.json; '0':'val/0.jpeg' for val_index.json and so on).
- Repacked train, val, folders + train_index.json, val_index.json into tinyimagenet_flat.tar and uploaded it to my student onedrive account.
- Save the link to download tinyimagenet_flat.tar to self.__datasets['tinyimagenet']['url']

For the places2 dataset:
- Downloaded the original archive from the provided link.
- Unpacked the archive locally.
- Originally, the dataset consists of two big train, val folders with many subfolders. For each train and val folder, i did not want to have subfolders, so i moved the images from the subfolders into the root train/val big folders and deleted all the subfolders. The final result are just the two big train and val folders with images inside, no subfolders.
- For each train, val, big folders i renamed the images with the following patters: 0.extension, 1.extension, 2.extension, etc. The reason behind is to make indexing easier.
Ex: for train: 0.extension, 1.extension, 2.extension,etc; for val: 0.extension, 1.extension, 2.extension. etc.
- The original archive had almost 30 GB in size. This is too big to use for this project, so i deleted images in both train and val subfolders, starting from the largest index(index.extension, where index - is the largest number) and counting backwards.
- Added train_index.json, val_index.json with a 'files' dictionary were key = the index('0', '1', '2','3', etc.), value = relative path to image('0':'train/0.jpeg for train_index.json; '0':'val/0.jpeg' for val_index.json and so on).
- Repacked train, val, folders + train_index.json, val_index.json into places2_reduced.tar. From the original 30GB archived, i got a final aprox 3GB .tar archive, uploaded it to my student onedrive account.
- Save the link to download tinyimagenet_flat.tar to self.__datasets['places2']['url']

The tinyimagenet dataset:
- train has 100000 images of 64x64 size
- val has 20000 images of 64x64 size
  
The place2_reduced dataset:
- train has 167000 images of 128x128 size
- val has 39000 images of 128x128 size

In both datasets a single image is in either train or val, not both.

In [38]:
class MyDataset(Dataset):
    def __init__(self, dataset='places2', split='train'):
        super().__init__()
        self.__root_dir = Path().cwd() # /content on google colab
        self.__datasets = {
            'places2': {
                'name': 'places2',
                'url': 'https://ctipub-my.sharepoint.com/:u:/g/personal/marius_vlad_dumitru_stud_etti_upb_ro/IQDIXTjG7MdBRrMizEF0KjaWAR46T_nR4YNLuvv3HThlCN4?download=1',
                'archive': self.__root_dir / 'places2_reduced.tar',
                'extract_dir': self.__root_dir / 'places2'
            },
            'tinyimagenet': {
                'name': 'tinyimagenet',
                'url': 'https://ctipub-my.sharepoint.com/:u:/g/personal/marius_vlad_dumitru_stud_etti_upb_ro/IQBkCh4mNvQ8Q7t70sitsGOyAWD29CnOXxthhkGPnbmeRBY?download=1',
                'archive': self.__root_dir / 'tinyimagenet_flat.tar',
                'extract_dir': self.__root_dir / 'tinyimagenet'
            }
        }

        self.change_dataset(dataset)
        self.change_split(split)

    def __download_dataset(self):
        try:
            print(f"[INFO] Downloading dataset {self.__dataset['name']}. This may take a whike ...")
            response = requests.get(self.__dataset['url'], stream=True)
            response.raise_for_status()
            total_size = int(response.headers.get("Content-Length", 0))
            with open(str(self.__dataset['archive']), "wb") as f, tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                desc="Downloading",
            ) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
            print(f"[INFO] Download complete: {str(self.__dataset['archive'])}")
            return True

        except Exception as e:
            print(f"[ERROR] Download failed: {e}")
            return False
    
    def __extract_dataset(self):
        try:
            print(f"[INFO] Extracting dataset {self.__dataset['archive']} into folder {self.__dataset['extract_dir']}. This may take a while ...")
            archive = self.__dataset['archive']
            out_dir = self.__dataset['extract_dir']
            out_dir.mkdir(parents=True, exist_ok=True)
            subprocess.run(
                ["tar", "-xf", str(archive), "-C", str(out_dir)],
                check=True
            )

            print(f"[INFO] Extraction complete: {out_dir}")
            return True

        except Exception as e:
            print(f"[ERROR] Extraction failed: {e}")
            return False
    
    def change_dataset(self, new_dataset):
        self.__dataset = self.__datasets[new_dataset] 
        if not self.__dataset['archive'].exists():
            self.__download_dataset()
        if not self.__dataset['extract_dir'].exists():
            self.__extract_dataset()
        
    def get_dataset(self):
        return self.__dataset['name']
        
    def change_split(self, new_split):
        self.__split = new_split
        json_name = self.__split + '_index.json'
        json_path =  self.__dataset['extract_dir'] / json_name
        with json_path.open() as f:
            self.__json_data = json.load(f)

    def get_split(self):
        return self.__split

    def __len__(self):
        return len(self.__json_data['files'])
    
    def __getitem__(self, idx):
        pass

In [39]:
# Helper print function for MyDataset class
def print_mydataset_info(mydataset):
    print()
    print(f'Current dataset is: {mydataset.get_dataset()}')
    print(f'Current split for {mydataset.get_dataset()} dataset is: {mydataset.get_split()}')
    print(f'Total number of images for dataset {mydataset.get_dataset()} with the split {mydataset.get_split()} is: {mydataset.__len__()}')

In [40]:
# Creating a MyDataset object
mydataset = MyDataset()
print_mydataset_info(mydataset)
print(f'Changing split to val for dataset {mydataset.get_dataset()}')
mydataset.change_split('val')
print_mydataset_info(mydataset)

[INFO] Downloading dataset places2. This may take a whike ...


Downloading:  36%|███▌      | 1.06G/2.99G [00:21<00:39, 48.7MB/s] 


KeyboardInterrupt: 

In [34]:
# Changing the active dataset
mydataset.change_dataset('tinyimagenet')
mydataset.change_split('train')
print_mydataset_info(mydataset)
print(f'Changing split to val for dataset {mydataset.get_dataset()}')
mydataset.change_split('val')
print_mydataset_info(mydataset)

[INFO] Downloading dataset tinyimagenet. This may take a whike ...


Downloading: 100%|██████████| 330M/330M [00:08<00:00, 37.0MB/s] 


[INFO] Download complete: /content/tinyimagenet_flat.tar
[INFO] Extracting dataset /content/tinyimagenet_flat.tar. This may take a while ...
[INFO] Extraction complete: /content/tinyimagenet

Current dataset is: tinyimagenet
Current split for tinyimagenet dataset is: train
Total number of images for dataset tinyimagenet with the split train is: 100000
Changing split to val for dataset tinyimagenet

Current dataset is: tinyimagenet
Current split for tinyimagenet dataset is: val
Total number of images for dataset tinyimagenet with the split val is: 20000


In [41]:
!rm -r ./*
!ls -la

total 16
drwxr-xr-x 1 root root 4096 Mar 26 11:43 .
drwxr-xr-x 1 root root 4096 Mar 26 10:25 ..
drwxr-xr-x 4 root root 4096 Mar 23 13:29 .config
